In [23]:
import os
import gzip
import gc
import pickle

from io import StringIO
from google.cloud import storage
from dotenv import load_dotenv

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [24]:
load_dotenv()
cred_path = os.getenv('GOOGLE_APPLICATION_CREDENTIALS')

os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = cred_path

client = storage.Client()
bucket = client.bucket('domainome-data')
blob = bucket.blob('SupplementaryTable2.txt')

df = pd.read_csv(StringIO(blob.download_as_text()), sep='\t')

In [25]:
df.head()

,domain_ID,uniprot_ID,aa_seq,wt_aa,position,mut_aa,STOP,input_count_rep1,input_count_rep2,input_count_rep3,output_count_rep1,output_count_rep2,output_count_rep3,mean_input_count,fitness,fitness_sigma,normalized_fitness,normalized_fitness_sigma,quality_rank
0,A0A2R8Y422_PF00240_2,A0A2R8Y422,*IFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,Q,2.0,*,True,118.0,113.0,62.0,10.0,29.0,2.0,97.66667,0.030945,0.014885,-0.819050,0.208478,339
1,A0A2R8Y422_PF00240_2,A0A2R8Y422,AIFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,Q,2.0,A,False,219.0,277.0,217.0,86.0,225.0,137.0,237.66670,0.069376,0.006673,-0.280790,0.093461,339
2,A0A2R8Y422_PF00240_2,A0A2R8Y422,CIFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,Q,2.0,C,False,706.0,726.0,459.0,768.0,507.0,616.0,630.33330,0.082052,0.004141,-0.103250,0.057995,339
3,A0A2R8Y422_PF00240_2,A0A2R8Y422,DIFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,Q,2.0,D,False,407.0,431.0,323.0,508.0,159.0,111.0,387.00000,0.071003,0.005162,-0.258003,0.072296,339
4,A0A2R8Y422_PF00240_2,A0A2R8Y422,EIFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,Q,2.0,E,False,37.0,56.0,37.0,201.0,102.0,95.0,43.33333,0.116326,0.012085,0.376783,0.169263,339


In [26]:
df = df[['domain_ID','aa_seq','wt_aa','position','mut_aa','normalized_fitness']]

In [27]:
df

,domain_ID,aa_seq,wt_aa,position,mut_aa,normalized_fitness
0,A0A2R8Y422_PF00240_2,*IFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,Q,2.0,*,-0.819050
1,A0A2R8Y422_PF00240_2,AIFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,Q,2.0,A,-0.280790
2,A0A2R8Y422_PF00240_2,CIFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,Q,2.0,C,-0.103250
3,A0A2R8Y422_PF00240_2,DIFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,Q,2.0,D,-0.258003
4,A0A2R8Y422_PF00240_2,EIFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,Q,2.0,E,0.376783
...,...,...,...,...,...,...
602877,Q9Y6V0_PF05715_1058,TWPLCKTELNIGSKDPPNFNTCTECKNQVCNLCGFNPTPHLTEIQE...,C,1059.0,W,-0.903822
602878,Q9Y6V0_PF05715_1058,TYPLCKTELNIGSKDPPNFNTCTECKNQVCNLCGFNPTPHLTEIQE...,C,1059.0,Y,-1.149495
602879,Q9Y6V0_PF05715_1058,VCPLCKTELNIGSKDPPNFNTCTECKNQVCNLCGFNPTPHLTEIQE...,T,1058.0,V,-0.201507
602880,Q9Y6V0_PF05715_1058,WCPLCKTELNIGSKDPPNFNTCTECKNQVCNLCGFNPTPHLTEIQE...,T,1058.0,W,-0.515568


In [30]:
def restore_wt_seq(row):
    if pd.isna(row["position"]):
        return None

    # extract domain start
    domain_start = int(row["domain_ID"].split("_")[2])
    
    # convert to 0-based local index
    pos = int(row["position"]) - domain_start

    seq = row["aa_seq"]

    if pos < 0 or pos >= len(seq):
        return None

    return seq[:pos] + row["wt_aa"] + seq[pos+1:]

In [31]:
df_one = df.drop_duplicates(subset=["domain_ID"]).copy()

df_one["wt_seq"] = df_one.apply(restore_wt_seq, axis=1)

In [35]:
df_one.set_index('domain_ID', inplace=True)

In [36]:
df_one.head()

,aa_seq,wt_aa,position,mut_aa,normalized_fitness,wt_seq
domain_ID,,,,,,
A0A2R8Y422_PF00240_2,*IFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,Q,2.0,*,-0.819050,QIFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...
A0PJY2_PF00096_289,*CKVCGKGFRQASTLCRHKIIH,V,289.0,*,NaN,VCKVCGKGFRQASTLCRHKIIH
A1X283_PF00018_155,*QYVVVANYQKQESSEISLSVGQVVDIIEKNESGWWFVSTAEEQGW...,E,155.0,*,NaN,EQYVVVANYQKQESSEISLSVGQVVDIIEKNESGWWFVSTAEEQGW...
A1X283_PF00018_222,*EEKYTVIYPYTARDQDEMNLERGAVVEVIQKNLEGWWKIRYQGKE...,E,222.0,*,-1.047782,EEEKYTVIYPYTARDQDEMNLERGAVVEVIQKNLEGWWKIRYQGKE...
A2RRE5_PF01846_267,*QQIATAKDKYEWLVSRIVKNHNENWLSVSRKMQASPEYQDYVYLE...,S,267.0,*,-0.984201,SQQIATAKDKYEWLVSRIVKNHNENWLSVSRKMQASPEYQDYVYLE...


In [33]:
len(df_one)

522

In [34]:
len(df['domain_ID'].unique())

522

In [37]:
aa_order = list("ACDEFGHIKLMNPQRSTVWY")

def build_dict(df):
    result = {}

    for domain_id, df_domain in df.groupby("domain_ID"):
        # get all positions that should exist
        # positions = sorted(df_domain["position"].dropna().unique())
        positions = df_domain["position"].dropna()
        min_pos, max_pos = int(positions.min()), int(positions.max())
        positions = list(range(min_pos, max_pos + 1))
        
        result[domain_id] = {}
        pos_lists = []

        for pos in positions:
            df_pos = df_domain[df_domain["position"] == pos]

            # map mut_aa -> fitness
            fitness_map = dict(zip(df_pos["mut_aa"], df_pos["normalized_fitness"]))

            # build ordered list, fill missing with NaN
            row = [fitness_map.get(aa, np.nan) for aa in aa_order]

            pos_lists.append(row)

        result[domain_id]['fitness'] = pos_lists
        result[domain_id]['uniprot_id'] = domain_id.split("_")[0]
        result[domain_id]['dom_seq'] = df_one.loc[domain_id]['wt_seq']

    return result

In [38]:
dict_fitness = build_dict(df)

In [41]:
dict_fitness.keys()

dict_keys(['A0A2R8Y422_PF00240_2', 'A0PJY2_PF00096_289', 'A1X283_PF00018_155', 'A1X283_PF00018_222', 'A2RRE5_PF01846_267', 'A6NK59_PF07525_532', 'EEHEE-rd3-0037_rockdoms_1', 'EHEE-rd1-0882_rockdoms_1', 'HHH-rd1-0142_rockdoms_1', 'O00308_PF00397_440', 'O00330_PF02817_181', 'O14529_PF02376_1036', 'O14529_PF02376_895', 'O14640_PF00595_246', 'O14640_PF00778_4', 'O14776_PF01846_645', 'O14776_PF01846_727', 'O14776_PF01846_793', 'O14776_PF01846_897', 'O14776_PF01846_955', 'O14813_PF00046_92', 'O14901_PF00096_395', 'O14936_PF00595_488', 'O15151_PF00641_299', 'O15259_PF00018_152', 'O15265_PF08313_334', 'O15266_PF00046_119', 'O15344_PF00643_172', 'O15350_PF07647_487', 'O15405_PF00505_256', 'O15541_PF00642_198', 'O43167_PF00096_351', 'O43167_PF00096_435', 'O43186_PF00046_41', 'O43295_PF00018_745', 'O43353_PF00619_438', 'O43374_PF00779_682', 'O43395_PF01480_10', 'O43586_PF00018_361', 'O60281_PF00096_808', 'O60293_PF10650_1186', 'O60341_PF04433_186', 'O60481_PF00096_389', 'O60885_PF17035_607', 'O75

In [43]:
df[df['domain_ID']=='O14776_PF01846_955']

,domain_ID,aa_seq,wt_aa,position,mut_aa,normalized_fitness
22139,O14776_PF01846_955,*KKREHFRQLLDETSAITLTSTWKEVKKIIKEDPRCIKFSSSDRKK...,K,955.0,*,NaN
22140,O14776_PF01846_955,AKKREHFRQLLDETSAITLTSTWKEVKKIIKEDPRCIKFSSSDRKK...,K,955.0,A,-0.579791
22141,O14776_PF01846_955,CKKREHFRQLLDETSAITLTSTWKEVKKIIKEDPRCIKFSSSDRKK...,K,955.0,C,NaN
22142,O14776_PF01846_955,DKKREHFRQLLDETSAITLTSTWKEVKKIIKEDPRCIKFSSSDRKK...,K,955.0,D,NaN
22143,O14776_PF01846_955,EKKREHFRQLLDETSAITLTSTWKEVKKIIKEDPRCIKFSSSDRKK...,K,955.0,E,-0.182228
...,...,...,...,...,...,...
23255,O14776_PF01846_955,SKKREHFRQLLDETSAITLTSTWKEVKKIIKEDPRCIKFSSSDRKK...,K,955.0,S,-0.223652
23256,O14776_PF01846_955,TKKREHFRQLLDETSAITLTSTWKEVKKIIKEDPRCIKFSSSDRKK...,K,955.0,T,-0.928017
23257,O14776_PF01846_955,VKKREHFRQLLDETSAITLTSTWKEVKKIIKEDPRCIKFSSSDRKK...,K,955.0,V,0.277028
23258,O14776_PF01846_955,WKKREHFRQLLDETSAITLTSTWKEVKKIIKEDPRCIKFSSSDRKK...,K,955.0,W,-0.397834


In [42]:
dict_fitness['O14776_PF01846_955']

{'fitness': [[-0.579790665196086,
   nan,
   nan,
   -0.182227612872974,
   nan,
   0.244230579632941,
   nan,
   nan,
   nan,
   -1.26874301459624,
   -0.223897056418318,
   nan,
   nan,
   -0.338761128239441,
   -0.462575583453117,
   -0.223652389693654,
   -0.928016549748226,
   0.2770281949883,
   -0.397833822286963,
   nan],
  [-0.312306027208061,
   0.374955437822491,
   0.181349594827608,
   0.0568199599572219,
   -0.0684428568037957,
   -0.144464654088651,
   -0.640025403443257,
   -0.51150238211901,
   nan,
   -0.408511601852644,
   -0.333376823730116,
   -0.0403659181527373,
   -0.591278381631574,
   -0.0367703810685465,
   -0.335098491919724,
   0.185827241373965,
   -0.623962664741562,
   -0.430457143286625,
   -0.95862591174206,
   -0.374812238234478],
  [-0.483019574921351,
   -0.803478650215903,
   -0.159061192665754,
   -0.303772150045789,
   -0.34633761387042,
   -0.338731670038478,
   -0.0609261058581026,
   -0.327691390944282,
   nan,
   -0.206544539484465,
   -0.744

In [44]:
load_dotenv()
cred_path = os.getenv('GOOGLE_APPLICATION_CREDENTIALS')

os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = cred_path

client = storage.Client()
bucket = client.bucket('domainome-data')

data = dict_fitness
compressed_data = gzip.compress(pickle.dumps(data))

blob_name = "dict_fitness.pkl.gz"
blob = bucket.blob("ESM2/"+blob_name)

blob.upload_from_string(compressed_data)

In [22]:
fitn = dict_fitness['A0A2R8Y422_PF00240_2']
for i in range(len(fitn)):
    print(fitn[i])

[-0.280789669693156, -0.103250266182212, -0.258003463940259, 0.376782636488238, 0.149046632098227, -0.187511607393213, -0.111728040072573, 0.28014749897967, 0.179340002571484, 0.133359739131628, -0.170486170149888, 0.213562029209033, -0.728559271446471, nan, -0.06181253769339, -0.040570763010855, 0.160116898879892, 0.0749988165010841, -0.203489963228899, 0.267453947886271]
[-0.658535772340188, -0.458064361612281, -0.955485314389134, -1.48141990737615, -0.252340176524816, -1.0048814778358, -1.00963634229646, nan, -1.38335060728201, -0.0586843189615014, 0.0091776489296827, -1.18316759213014, -0.869488221454636, -1.1257808641811, -1.02035926265076, -1.09083221122697, -0.626917163199318, -0.268435061484519, -0.526229277214102, -0.422089496047954]
[-0.0531473648031064, -0.0472066204229054, -0.0222738697725414, -0.100256224013795, nan, -0.219191702454087, -0.134904310261034, 0.289581876135704, -0.0288922198041806, -0.0614616897899479, -0.226416508143173, 0.0269285920172507, -0.86988010665776

In [11]:
df[df['domain_ID']=='A0PJY2_PF00096_289'][['aa_seq','position','mut_aa','normalized_fitness']][0:40]

,aa_seq,position,mut_aa,normalized_fitness
1368,ACKVCGKGFRQASTLCRHKIIH,289.0,A,-0.252455
1369,CCKVCGKGFRQASTLCRHKIIH,289.0,C,-0.801103
1370,DCKVCGKGFRQASTLCRHKIIH,289.0,D,0.213685
1371,ECKVCGKGFRQASTLCRHKIIH,289.0,E,-0.170356
1372,FCKVCGKGFRQASTLCRHKIIH,289.0,F,-0.314744
1373,GCKVCGKGFRQASTLCRHKIIH,289.0,G,-0.116247
1374,HCKVCGKGFRQASTLCRHKIIH,289.0,H,-0.202370
1375,ICKVCGKGFRQASTLCRHKIIH,289.0,I,0.215031
1376,KCKVCGKGFRQASTLCRHKIIH,289.0,K,-0.413446
1377,LCKVCGKGFRQASTLCRHKIIH,289.0,L,-0.280329


In [8]:
df = df.dropna(subset=["normalized_fitness"]).reset_index(drop=True)

In [9]:
df

,domain_ID,uniprot_ID,aa_seq,wt_aa,position,mut_aa,STOP,input_count_rep1,input_count_rep2,input_count_rep3,output_count_rep1,output_count_rep2,output_count_rep3,mean_input_count,fitness,fitness_sigma,normalized_fitness,normalized_fitness_sigma,quality_rank
0,A0A2R8Y422_PF00240_2,A0A2R8Y422,*IFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,Q,2.0,*,True,118.0,113.0,62.0,10.0,29.0,2.0,97.66667,0.030945,0.014885,-0.819050,0.208478,339
1,A0A2R8Y422_PF00240_2,A0A2R8Y422,AIFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,Q,2.0,A,False,219.0,277.0,217.0,86.0,225.0,137.0,237.66670,0.069376,0.006673,-0.280790,0.093461,339
2,A0A2R8Y422_PF00240_2,A0A2R8Y422,CIFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,Q,2.0,C,False,706.0,726.0,459.0,768.0,507.0,616.0,630.33330,0.082052,0.004141,-0.103250,0.057995,339
3,A0A2R8Y422_PF00240_2,A0A2R8Y422,DIFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,Q,2.0,D,False,407.0,431.0,323.0,508.0,159.0,111.0,387.00000,0.071003,0.005162,-0.258003,0.072296,339
4,A0A2R8Y422_PF00240_2,A0A2R8Y422,EIFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,Q,2.0,E,False,37.0,56.0,37.0,201.0,102.0,95.0,43.33333,0.116326,0.012085,0.376783,0.169263,339
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
563529,Q9Y6V0_PF05715_1058,Q9Y6V0,TVPLCKTELNIGSKDPPNFNTCTECKNQVCNLCGFNPTPHLTEIQE...,C,1059.0,V,False,146.0,118.0,99.0,10.0,22.0,8.0,121.00000,0.009509,0.013942,-1.113722,0.157535,380
563530,Q9Y6V0_PF05715_1058,Q9Y6V0,TWPLCKTELNIGSKDPPNFNTCTECKNQVCNLCGFNPTPHLTEIQE...,C,1059.0,W,False,118.0,121.0,83.0,13.0,47.0,4.0,107.33330,0.028085,0.013317,-0.903822,0.150480,380
563531,Q9Y6V0_PF05715_1058,Q9Y6V0,TYPLCKTELNIGSKDPPNFNTCTECKNQVCNLCGFNPTPHLTEIQE...,C,1059.0,Y,False,177.0,168.0,126.0,12.0,6.0,29.0,157.00000,0.006343,0.013753,-1.149495,0.155403,380
563532,Q9Y6V0_PF05715_1058,Q9Y6V0,VCPLCKTELNIGSKDPPNFNTCTECKNQVCNLCGFNPTPHLTEIQE...,T,1058.0,V,False,20.0,19.0,23.0,46.0,43.0,8.0,20.66667,0.090240,0.021806,-0.201507,0.246396,380
